## Configuration

In [ ]:
# CONFIG  -- edit these values

XML_PATH = r"C:\\path\\to\\trace.xml"
OUTPUT_PATH = r"C:\\path\\to\\output"
OUTPUT_FILE = "trace_search.txt"

SEARCH_TEXT = "00000000-0000-0000-0000-000000000000"

# Event classes to drop as connection-lifecycle noise.
SKIP_EVENT_CLASSES = {
    'audit login', 'audit logout', 'audit login failed',
    'existingconnection', 'login', 'logout',
}

## Imports

In [ ]:
import re
import os
from collections import defaultdict, OrderedDict
from xml.etree import ElementTree as ET

import sqlparse

## Encoding detection

In [ ]:
INVALID_CHARREF = re.compile(r'&#(?:x([0-9a-fA-F]+)|([0-9]+));')


def _is_valid_xml_char(codepoint):
    return (
        codepoint in (0x09, 0x0A, 0x0D)
        or 0x20    <= codepoint <= 0xD7FF
        or 0xE000  <= codepoint <= 0xFFFD
        or 0x10000 <= codepoint <= 0x10FFFF
    )


def _scrub_charref(match):
    hex_part, dec_part = match.group(1), match.group(2)
    try:
        cp = int(hex_part, 16) if hex_part else int(dec_part)
    except ValueError:
        return ''
    return match.group(0) if _is_valid_xml_char(cp) else ''


def diagnose_file(path):
    """Sniff the encoding from the first bytes and report what was found."""
    with open(path, 'rb') as f:
        head = f.read(512)

    print(f"First 32 raw bytes: {head[:32]!r}")

    if head.startswith(b'\xef\xbb\xbf'):
        print("Detected: UTF-8 with BOM")
        return 'utf-8-sig'
    if head.startswith(b'\xff\xfe'):
        print("Detected: UTF-16 LE with BOM")
        return 'utf-16'
    if head.startswith(b'\xfe\xff'):
        print("Detected: UTF-16 BE with BOM")
        return 'utf-16'
    if head[:20].count(b'\x00') > 5:
        print("Detected: UTF-16 (no BOM, inferred from null bytes)")
        return 'utf-16'

    head_text = head.lstrip().decode('utf-8', errors='replace')
    if not head_text.startswith('<'):
        print("WARNING: file does not start with '<'. Probably not XML.")
        print(f"First chars: {head_text[:80]!r}")
        return None

    print("Detected: UTF-8 (no BOM)")
    return 'utf-8'


def load_and_sanitize_xml(path):
    encoding = diagnose_file(path)
    if encoding is None:
        raise ValueError("File does not appear to be XML.")

    print(f"Reading with encoding: {encoding}")
    with open(path, 'r', encoding=encoding, errors='replace') as f:
        raw = f.read()

    if raw.startswith('\ufeff'):
        raw = raw[1:]

    cleaned = INVALID_CHARREF.sub(_scrub_charref, raw)
    cleaned = ''.join(ch for ch in cleaned if _is_valid_xml_char(ord(ch)))

    return ET.fromstring(cleaned)

## Event extraction

In [ ]:
def get_columns(event):
    out = {}
    for col in event:
        if col.tag.split('}')[-1] != 'Column':
            continue
        name = (col.attrib.get('name') or '').lower()
        val  = (col.text or '').strip()
        out[name] = val
    return out


def get_xe_fields(event):
    out = {}
    for child in event:
        tag = child.tag.split('}')[-1]
        if tag not in ('data', 'action'):
            continue
        name = (child.attrib.get('name') or '').lower()
        value_node = None
        for sub in child:
            if sub.tag.split('}')[-1] == 'value':
                value_node = sub
                break
        val = (value_node.text or '').strip() if value_node is not None and value_node.text else ''
        out[name] = val
    return out


def collect_hits(root, needle):
    hits      = []
    needle_lc = needle.lower()

    for event in root.iter():
        tag = event.tag.split('}')[-1]

        if tag == 'Event':
            cols        = get_columns(event)
            event_class = (event.attrib.get('name') or '').strip()
            text_data   = cols.get('textdata')
            if not text_data or needle_lc not in text_data.lower():
                continue
            hits.append({
                'spid':        cols.get('spid'),
                'event_class': event_class,
                'timestamp':   cols.get('starttime') or cols.get('endtime'),
                'login_name':  cols.get('loginname'),
                'host_name':   cols.get('hostname'),
                'application': cols.get('applicationname'),
                'database':    cols.get('databasename'),
                'text_data':   text_data,
            })

        elif tag == 'event':
            fields      = get_xe_fields(event)
            event_class = (event.attrib.get('name') or '').strip()
            text_data   = (fields.get('batch_text')
                           or fields.get('statement')
                           or fields.get('sql_text'))
            if not text_data or needle_lc not in text_data.lower():
                continue
            hits.append({
                'spid':        fields.get('session_id') or fields.get('spid'),
                'event_class': event_class,
                'timestamp':   event.attrib.get('timestamp'),
                'login_name':  (fields.get('server_principal_name')
                                or fields.get('username')
                                or fields.get('nt_username')),
                'host_name':   fields.get('client_hostname') or fields.get('host_name'),
                'application': fields.get('client_app_name'),
                'database':    fields.get('database_name'),
                'text_data':   text_data,
            })

    return hits

## Formatting helpers

In [ ]:
def consolidate_session_info(events):
    keys   = ('login_name', 'host_name', 'application', 'database')
    shared = {}
    for k in keys:
        vals = {(e.get(k) or '').strip() for e in events if e.get(k)}
        if len(vals) == 1:
            shared[k] = next(iter(vals))
        elif len(vals) > 1:
            shared[k] = '[varies: ' + ', '.join(sorted(vals)) + ']'
    return shared


def format_sql(raw_sql):
    """Consistently indented, keyword-uppercased SQL, with blank edge lines trimmed."""
    formatted = sqlparse.format(
        raw_sql,
        reindent        = True,
        keyword_case    = 'upper',
        identifier_case = 'lower',
        indent_width    = 4,
        strip_comments  = False,
    )
    lines = formatted.splitlines()
    while lines and not lines[0].strip():
        lines.pop(0)
    while lines and not lines[-1].strip():
        lines.pop()
    return '\n'.join(lines)

## Run the search

In [ ]:
def run_search():
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    output_file_path = os.path.join(OUTPUT_PATH, OUTPUT_FILE)

    lines_out = []

    def w(text=""):
        lines_out.append(text)

    w(f"Parsing  : {XML_PATH}")
    w(f"Searching: {SEARCH_TEXT}\n")

    try:
        root = load_and_sanitize_xml(XML_PATH)
    except (ET.ParseError, ValueError) as exc:
        w(f"\nXML load failed: {exc}")
        w("\nIf the built-in parser can't cope, try the lxml recover mode:")
        w("  pip install lxml")
        w("  from lxml import etree")
        w("  parser = etree.XMLParser(recover=True, huge_tree=True)")
        w("  root   = etree.parse(XML_PATH, parser).getroot()")
        _flush(output_file_path, lines_out)
        return

    hits = collect_hits(root, SEARCH_TEXT)

    if not hits:
        w("No matches found.")
        _flush(output_file_path, lines_out)
        return

    class_counts = defaultdict(int)
    for h in hits:
        class_counts[h['event_class'] or '(none)'] += 1

    w("Event class breakdown (before filtering):")
    for cls, n in sorted(class_counts.items(), key=lambda kv: -kv[1]):
        marker = '  [SKIPPED]' if cls.lower() in SKIP_EVENT_CLASSES else ''
        w(f"  {n:6d}  {cls}{marker}")
    w()

    filtered = [h for h in hits
                if (h['event_class'] or '').lower() not in SKIP_EVENT_CLASSES]

    if not filtered:
        w("All events were filtered out as lifecycle noise.")
        _flush(output_file_path, lines_out)
        return

    by_spid = defaultdict(list)
    for h in filtered:
        by_spid[h['spid']].append(h)

    def spid_sort_key(s):
        try:
            return (0, int(s))
        except (TypeError, ValueError):
            return (1, str(s))

    sorted_spids = sorted(by_spid.keys(), key=spid_sort_key)

    w(f"Found {len(filtered)} relevant event(s) across {len(by_spid)} SPID(s).")
    w(f"Distinct SPIDs: {sorted_spids}\n")

    for spid in sorted_spids:
        events = by_spid[spid]
        shared = consolidate_session_info(events)

        w('=' * 72)
        w(f"SPID: {spid}   ({len(events)} event(s))")
        w('=' * 72)

        if shared.get('login_name'):   w(f"  login      : {shared['login_name']}")
        if shared.get('host_name'):    w(f"  host       : {shared['host_name']}")
        if shared.get('application'):  w(f"  application: {shared['application']}")
        if shared.get('database'):     w(f"  database   : {shared['database']}")
        w()

        unique_sql = OrderedDict()
        for e in events:
            sql = (e['text_data'] or '').strip()
            if not sql:
                continue
            if sql not in unique_sql:
                unique_sql[sql] = {
                    'first_seen':    e['timestamp'],
                    'count':         1,
                    'event_classes': {e['event_class']},
                }
            else:
                unique_sql[sql]['count'] += 1
                unique_sql[sql]['event_classes'].add(e['event_class'])

        w(f"  Unique SQL statements: {len(unique_sql)}")
        w('  ' + '-' * 70)

        for i, (sql, meta) in enumerate(unique_sql.items(), 1):
            ts      = meta['first_seen'] or '(no timestamp)'
            classes = ', '.join(sorted(c for c in meta['event_classes'] if c))
            cnt     = meta['count']
            cnt_str = f"  [seen {cnt}x]" if cnt > 1 else ''

            w(f"\n  [{i}] {ts}   ({classes}){cnt_str}")

            for line in format_sql(sql).splitlines():
                w(f"      {line}")

        w()

    _flush(output_file_path, lines_out)


def _flush(output_file_path, lines_out):
    text = "\n".join(lines_out) + "\n"
    with open(output_file_path, "w", encoding="utf-8") as out:
        out.write(text)
    print(text)
    print(f"Output written to: {output_file_path}")


run_search()